# Experiment 20: Score Ensemble Analysis

This notebook performs a post-hoc score ensemble study using saved artifacts. It does **not** train any new models.

Initial focus:

- `PatchCore-WideRes50-topk-mb50k-r010`
- `TS-Res50-s2_a1_topk_mean_r0.20`

The notebook recomputes the selected TS-Res50 score from the saved checkpoint, loads the saved PatchCore scores, normalizes the component scores on validation normals, evaluates several fusion rules, and exports the best ensemble defect breakdown.


## Submission Context

- Dataset notebook: `data/dataset/x64/benchmark_50k_5pct/notebook.ipynb`
- Dataset config: `data/dataset/x64/benchmark_50k_5pct/data_config.toml`
- Experiment config: `embedded in notebook`
- Artifact root: `experiments/anomaly_detection/ensemble/x64/score_ensemble/artifacts/ensemble_patchcore_ts_res50`
- Notes:
  - This notebook does not train a model from scratch; it combines saved score files from the selected PatchCore and teacher-student branches.


In [3]:
from pathlib import Path
import json
import sys
from IPython.display import display
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.evaluation.reconstruction_metrics import summarize_threshold_metrics, sweep_threshold_metrics
from wafer_defect.models.ts_distillation import build_ts_distillation_from_config
from wafer_defect.scoring import spatial_max, spatial_mean, topk_spatial_mean
REPO_ROOT


WindowsPath('C:/Users/genso/Documents/College_Projects/DeepLearning 2610/Project')

In [4]:
RETRAIN = False
print(f'RETRAIN = {RETRAIN}')


RETRAIN = False


In [5]:
CONFIG = {
    'data': {
        'metadata_csv': REPO_ROOT / 'data' / 'processed' / 'x64' / 'wm811k' / 'metadata_50k_5pct.csv',
        'image_size': 64,
        'batch_size': 16,
        'num_workers': 0,
    },
    'patchcore': {
        'name': 'PatchCore-WideRes50-topk-mb50k-r010',
    },
    'ts_res50': {
        'name': 'TS-Res50-s2_a1_topk_mean_r0.20',
        'artifact_dir': REPO_ROOT / 'experiments/anomaly_detection/teacher_student/resnet50/x64/main/artifacts/ts_resnet50',
        'config_path': REPO_ROOT / 'experiments/anomaly_detection/teacher_student/resnet50/x64/main/train_config.toml',
        'selected_variant_json': REPO_ROOT / 'experiments/anomaly_detection/teacher_student/resnet50/x64/main/artifacts/ts_resnet50/results/evaluation/selected_score_variant.json',
        'converted_checkpoint_path': REPO_ROOT / 'experiments/anomaly_detection/teacher_student/resnet50/x64/main/artifacts/ts_resnet50/checkpoints/best_model.pt',
        'raw_checkpoint_path': REPO_ROOT / 'experiments/anomaly_detection/teacher_student/resnet50/x64/main/artifacts/ts_resnet50/checkpoints/best_model.pt',
    },
    'ensemble': {
        'threshold_quantile': 0.95,
        'fusion_variants': [
            {'name': 'equal_weight_mean', 'kind': 'weighted_mean', 'weights': {'patchcore': 1.0, 'ts_res50': 1.0}},
            {'name': 'patchcore_heavy', 'kind': 'weighted_mean', 'weights': {'patchcore': 2.0, 'ts_res50': 1.0}},
            {'name': 'ts_heavy', 'kind': 'weighted_mean', 'weights': {'patchcore': 1.0, 'ts_res50': 2.0}},
            {'name': 'max', 'kind': 'max'},
        ],
    },
}

def warn_skip(message: str) -> None:
    print(f'[WARNING] {message}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
output_dir = REPO_ROOT / 'experiments' / 'anomaly_detection' / 'ensemble' / 'x64' / 'score_ensemble' / 'artifacts'
output_dir.mkdir(parents=True, exist_ok=True)

def resolve_patchcore_score_paths(repo_root: Path) -> tuple[Path | None, Path | None]:
    preferred_root = repo_root / 'experiments/anomaly_detection/patchcore/wideresnet50/x64/main/artifacts/patchcore_wideresnet50_multilayer/topk_mb50k_r010/results/evaluation'
    legacy_candidates = sorted((repo_root / 'artifacts/x64').glob('*/topk_mb50k_r010'))
    for candidate in [preferred_root, *legacy_candidates]:
        val_scores = candidate / 'val_scores.csv'
        test_scores = candidate / 'test_scores.csv'
        if val_scores.exists() and test_scores.exists():
            return (val_scores, test_scores)
    return (None, None)

patchcore_val_path, patchcore_test_path = resolve_patchcore_score_paths(REPO_ROOT)
CONFIG['patchcore']['val_scores_csv'] = patchcore_val_path
CONFIG['patchcore']['test_scores_csv'] = patchcore_test_path
component_scores_ready = False
component_val_df = pd.DataFrame()
component_test_df = pd.DataFrame()
print(f'Results dir: {output_dir}')


Results dir: C:\Users\genso\Documents\College_Projects\DeepLearning 2610\Project\experiments\anomaly_detection\ensemble\x64\score_ensemble\artifacts


In [6]:
metadata_path = CONFIG['data']['metadata_csv']
component_val_path = output_dir / 'component_val_scores.csv'
component_test_path = output_dir / 'component_test_scores.csv'

if component_val_path.exists() and component_test_path.exists():
    component_val_df = pd.read_csv(component_val_path)
    component_test_df = pd.read_csv(component_test_path)
    component_scores_ready = not component_val_df.empty and not component_test_df.empty
    if component_scores_ready:
        print(f'Loaded cached ensemble component scores from {output_dir}')
        display(component_val_df.head())
        display(component_test_df.head())
else:
    patchcore_ready = metadata_path.exists() and CONFIG['patchcore']['val_scores_csv'] is not None and CONFIG['patchcore']['test_scores_csv'] is not None
    ts_cfg = CONFIG['ts_res50']
    ts_ready = ts_cfg['config_path'].exists() and ts_cfg['selected_variant_json'].exists() and (ts_cfg['converted_checkpoint_path'].exists() or ts_cfg['raw_checkpoint_path'].exists())
    if not metadata_path.exists():
        warn_skip(f'Benchmark metadata is missing: {metadata_path}. Skipping ensemble computation for this notebook run.')
    elif not patchcore_ready:
        warn_skip('PatchCore component score files are missing. Restore the WRN x64 PatchCore artifacts to enable this ensemble notebook.')
    elif not ts_ready:
        warn_skip('Teacher-student component artifacts are missing. Restore the TS ResNet50 x64 artifacts to enable this ensemble notebook.')
    else:
        metadata = pd.read_csv(metadata_path)
        val_metadata = metadata[metadata['split'] == 'val'].reset_index(drop=True)
        test_metadata = metadata[metadata['split'] == 'test'].reset_index(drop=True)
        patchcore_val_df = pd.read_csv(CONFIG['patchcore']['val_scores_csv']).reset_index(drop=True)
        patchcore_test_df = pd.read_csv(CONFIG['patchcore']['test_scores_csv']).reset_index(drop=True)
        if len(patchcore_val_df) != len(val_metadata) or len(patchcore_test_df) != len(test_metadata):
            warn_skip('PatchCore score CSV lengths do not match the metadata split sizes. Skipping ensemble computation.')
        else:
            def load_ts_checkpoint_for_local_eval(checkpoint_path: Path, config_path: Path, image_size: int):
                config = load_toml(config_path)
                model = build_ts_distillation_from_config(config, image_size=image_size)
                checkpoint = torch.load(checkpoint_path, map_location='cpu')
                remapped_state = dict(checkpoint['model_state_dict'])
                if 'auto_map_scale' in remapped_state:
                    remapped_state['autoencoder_map_scale'] = remapped_state.pop('auto_map_scale')
                layer_aliases = [('teacher.layer1.', 'teacher.layers.0.'), ('teacher.layer2.', 'teacher.layers.1.'), ('teacher.layer3.', 'teacher.layers.2.'), ('teacher.layer4.', 'teacher.layers.3.')]
                for old_prefix, new_prefix in layer_aliases:
                    for key, value in list(remapped_state.items()):
                        if key.startswith(old_prefix):
                            remapped_state[new_prefix + key[len(old_prefix):]] = value
                model.load_state_dict(remapped_state, strict=True)
                converted_checkpoint = {'epoch': int(checkpoint.get('best_epoch', 0)), 'model_state_dict': model.state_dict(), 'config': config, 'best_epoch': int(checkpoint.get('best_epoch', 0)), 'best_val_loss': float(checkpoint.get('best_val_loss', 0.0)), 'history': checkpoint.get('history', []), 'student_map_scale': float(checkpoint.get('student_map_scale', checkpoint.get('train_summary', {}).get('student_map_scale', 1.0))), 'autoencoder_map_scale': float(checkpoint.get('auto_map_scale', checkpoint.get('train_summary', {}).get('auto_map_scale', 1.0)))}
                return (converted_checkpoint, model)

            ts_cfg = CONFIG['ts_res50']
            if ts_cfg['converted_checkpoint_path'].exists():
                converted_checkpoint = torch.load(ts_cfg['converted_checkpoint_path'], map_location='cpu')
                ts_config = load_toml(ts_cfg['config_path'])
                ts_model = build_ts_distillation_from_config(ts_config, image_size=int(ts_config['data'].get('image_size', 64)))
                ts_model.load_state_dict(converted_checkpoint['model_state_dict'], strict=True)
            else:
                converted_checkpoint, ts_model = load_ts_checkpoint_for_local_eval(ts_cfg['raw_checkpoint_path'], ts_cfg['config_path'], image_size=CONFIG['data']['image_size'])
                torch.save(converted_checkpoint, ts_cfg['converted_checkpoint_path'])

            selected_variant = json.loads(ts_cfg['selected_variant_json'].read_text(encoding='utf-8'))
            ts_config = load_toml(ts_cfg['config_path'])
            ts_image_size = int(ts_config['data'].get('image_size', CONFIG['data']['image_size']))
            ts_batch_size = int(ts_config['data'].get('batch_size', CONFIG['data']['batch_size']))
            val_dataset = WaferMapDataset(metadata_path, split='val', image_size=ts_image_size)
            test_dataset = WaferMapDataset(metadata_path, split='test', image_size=ts_image_size)
            val_loader = DataLoader(val_dataset, batch_size=ts_batch_size, shuffle=False, num_workers=0)
            test_loader = DataLoader(test_dataset, batch_size=ts_batch_size, shuffle=False, num_workers=0)
            ts_model.to(device)
            ts_model.eval()

            def collect_normalized_maps(model, dataloader, device):
                student_maps = []
                auto_maps = []
                labels = []
                with torch.inference_mode():
                    for inputs, batch_labels in dataloader:
                        inputs = inputs.to(device)
                        student_map, auto_map = model.raw_anomaly_maps(inputs)
                        student_maps.append((student_map / model.student_map_scale.clamp_min(1e-06)).cpu())
                        auto_maps.append((auto_map / model.autoencoder_map_scale.clamp_min(1e-06)).cpu())
                        labels.append(batch_labels.cpu())
                return (torch.cat(student_maps, dim=0), torch.cat(auto_maps, dim=0), torch.cat(labels, dim=0).numpy())

            def reduce_anomaly_map(anomaly_map, reduction, topk_ratio):
                if reduction == 'mean':
                    return spatial_mean(anomaly_map)
                if reduction == 'max':
                    return spatial_max(anomaly_map)
                return topk_spatial_mean(anomaly_map, topk_ratio=topk_ratio)

            val_student_maps, val_auto_maps, _ = collect_normalized_maps(ts_model, val_loader, device)
            test_student_maps, test_auto_maps, _ = collect_normalized_maps(ts_model, test_loader, device)
            student_weight = float(selected_variant['student_weight'])
            auto_weight = float(selected_variant['auto_weight'])
            reduction = str(selected_variant['reduction'])
            topk_ratio_value = selected_variant.get('topk_ratio', np.nan)
            topk_ratio = None if pd.isna(topk_ratio_value) else float(topk_ratio_value)
            ts_val_scores = reduce_anomaly_map(student_weight * val_student_maps + auto_weight * val_auto_maps, reduction, topk_ratio)
            ts_test_scores = reduce_anomaly_map(student_weight * test_student_maps + auto_weight * test_auto_maps, reduction, topk_ratio)
            if hasattr(ts_val_scores, 'numpy'):
                ts_val_scores = ts_val_scores.numpy()
            if hasattr(ts_test_scores, 'numpy'):
                ts_test_scores = ts_test_scores.numpy()
            component_val_df = pd.DataFrame({'patchcore': patchcore_val_df['score'].to_numpy(), 'ts_res50': np.asarray(ts_val_scores).reshape(-1), 'is_anomaly': val_metadata['is_anomaly'].to_numpy().astype(int)})
            component_test_df = pd.DataFrame({'patchcore': patchcore_test_df['score'].to_numpy(), 'ts_res50': np.asarray(ts_test_scores).reshape(-1), 'is_anomaly': test_metadata['is_anomaly'].to_numpy().astype(int)})
            component_val_df.to_csv(component_val_path, index=False)
            component_test_df.to_csv(component_test_path, index=False)
            component_scores_ready = True
            display(component_val_df.head())
            display(component_test_df.head())


[WARNING] PatchCore component score files are missing. Restore the WRN x64 PatchCore artifacts to enable this ensemble notebook.


In [7]:
if not (component_scores_ready):
    print('[WARNING] Ensemble scoring is unavailable because the local PatchCore or teacher-student artifacts are missing.')
else:
    def zscore_against_val_normals(val_scores: np.ndarray, test_scores: np.ndarray, val_labels: np.ndarray):
        normal_scores = val_scores[val_labels == 0]
        mean = float(normal_scores.mean())
        std = float(normal_scores.std())
        std = max(std, 1e-08)
        return ((val_scores - mean) / std, (test_scores - mean) / std, mean, std)

    def fuse_scores(variant: dict, score_map: dict[str, np.ndarray]) -> np.ndarray:
        if variant['kind'] == 'weighted_mean':
            weights = variant['weights']
            total = sum((float(v) for v in weights.values()))
            if total <= 0:
                raise ValueError(f"Fusion weights must sum to > 0 for {variant['name']}")
            fused = sum((float(weights[k]) * score_map[k] for k in weights)) / total
            return np.asarray(fused).reshape(-1)
        if variant['kind'] == 'max':
            stacked = np.column_stack([score_map['patchcore'], score_map['ts_res50']])
            return stacked.max(axis=1)
        raise ValueError(f"Unsupported fusion kind: {variant['kind']}")

    def build_defect_breakdown(metadata_df: pd.DataFrame, scores: np.ndarray, threshold: float, comparator: str='>') -> pd.DataFrame:
        analysis_df = metadata_df.reset_index(drop=True).copy()
        analysis_df['score'] = np.asarray(scores).reshape(-1)
        if comparator == '>=':
            analysis_df['predicted_anomaly'] = (analysis_df['score'] >= threshold).astype(int)
        else:
            analysis_df['predicted_anomaly'] = (analysis_df['score'] > threshold).astype(int)
        defect_breakdown_df = analysis_df.loc[analysis_df['is_anomaly'] == 1].groupby('defect_type').agg(count=('defect_type', 'size'), detected=('predicted_anomaly', 'sum'), mean_score=('score', 'mean'), median_score=('score', 'median')).reset_index()
        defect_breakdown_df['detected'] = defect_breakdown_df['detected'].astype(int)
        defect_breakdown_df['missed'] = defect_breakdown_df['count'] - defect_breakdown_df['detected']
        defect_breakdown_df['recall'] = defect_breakdown_df['detected'] / defect_breakdown_df['count']
        return defect_breakdown_df.sort_values(['recall', 'count', 'defect_type'], ascending=[True, False, True]).reset_index(drop=True)
    val_labels_np = component_val_df['is_anomaly'].to_numpy().astype(int)
    test_labels_np = component_test_df['is_anomaly'].to_numpy().astype(int)
    normalized = {}
    normalization_rows = []
    for model_name in ['patchcore', 'ts_res50']:
        z_val, z_test, mean, std = zscore_against_val_normals(component_val_df[model_name].to_numpy(), component_test_df[model_name].to_numpy(), val_labels_np)
        normalized[model_name] = {'val': z_val, 'test': z_test}
        normalization_rows.append({'model': model_name, 'val_normal_mean': mean, 'val_normal_std': std})
    normalization_df = pd.DataFrame(normalization_rows)
    normalization_df.to_csv(output_dir / 'normalization_stats.csv', index=False)
    ensemble_rows = []
    best_variant_name = None
    best_scores = None
    best_threshold = None
    best_breakdown_df = None
    best_f1 = -1.0
    for variant in CONFIG['ensemble']['fusion_variants']:
        fused_val = fuse_scores(variant, {k: normalized[k]['val'] for k in normalized})
        fused_test = fuse_scores(variant, {k: normalized[k]['test'] for k in normalized})
        threshold = float(pd.Series(fused_val[val_labels_np == 0]).quantile(CONFIG['ensemble']['threshold_quantile']))
        metrics = summarize_threshold_metrics(test_labels_np, fused_test, threshold)
        sweep_df, best_sweep = sweep_threshold_metrics(test_labels_np, fused_test)
        row = {'name': variant['name'], 'kind': variant['kind'], 'threshold': threshold, 'precision': float(metrics['precision']), 'recall': float(metrics['recall']), 'f1': float(metrics['f1']), 'auroc': float(metrics['auroc']), 'auprc': float(metrics['auprc']), 'predicted_anomalies': int(metrics['predicted_anomalies']), 'best_sweep_f1': float(best_sweep['f1'])}
        if variant['kind'] == 'weighted_mean':
            row['patchcore_weight'] = float(variant['weights']['patchcore'])
            row['ts_res50_weight'] = float(variant['weights']['ts_res50'])
        ensemble_rows.append(row)
        sweep_df.to_csv(output_dir / f"threshold_sweep_{variant['name']}.csv", index=False)
        if row['f1'] > best_f1:
            best_f1 = row['f1']
            best_variant_name = variant['name']
            best_scores = fused_test
            best_threshold = threshold
            best_breakdown_df = build_defect_breakdown(test_metadata, fused_test, threshold, comparator='>')
    ensemble_df = pd.DataFrame(ensemble_rows).sort_values(['f1', 'auprc', 'auroc'], ascending=False).reset_index(drop=True)
    ensemble_df.to_csv(output_dir / 'ensemble_results.csv', index=False)
    best_scores_df = test_metadata.reset_index(drop=True).copy()
    best_scores_df['score'] = np.asarray(best_scores).reshape(-1)
    best_scores_df['predicted_anomaly'] = (best_scores_df['score'] > best_threshold).astype(int)
    best_scores_df.to_csv(output_dir / f'{best_variant_name}_test_scores.csv', index=False)
    best_breakdown_df.to_csv(output_dir / f'{best_variant_name}_defect_breakdown.csv', index=False)
    (output_dir / 'best_ensemble_summary.json').write_text(json.dumps({'best_variant': best_variant_name, 'threshold': best_threshold, 'best_f1': best_f1}, indent=2), encoding='utf-8')
    display(normalization_df)
    display(ensemble_df)
    display(best_breakdown_df)
    print(f'Best ensemble variant: {best_variant_name}')
    print(f'Saved ensemble outputs to: {output_dir}')


[WARNING] Ensemble scoring is unavailable because the local PatchCore or teacher-student artifacts are missing.
